去除多余通道。将 `data/interim/PGD_dataset/HE_unscaled` 中的 RGBA 四通道图转为 RGB 三通道（丢掉无信息的 Alpha），覆盖原文件。已是三通道的文件跳过。

In [3]:
import os
import numpy as np
import tifffile

HE_unscaled_dir = "/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unscaled"

converted, skipped, failed = [], [], []

for fname in sorted(os.listdir(HE_unscaled_dir)):
    path = os.path.join(HE_unscaled_dir, fname)
    if not os.path.isfile(path) or not fname.lower().endswith((".tif", ".tiff")):
        continue

    tmp_path = path + ".rgb.tmp"
    try:
        with tifffile.TiffFile(path) as tif:
            page = tif.pages[0]
            shape = page.shape
            samples = page.samplesperpixel

        is_rgba = samples == 4 or (len(shape) == 3 and min(shape) == 4)
        if not is_rgba:
            skipped.append((fname, shape))
            print(f"[跳过] {fname}  shape={shape}")
            continue

        img = tifffile.imread(path)
        if img.ndim == 3 and img.shape[-1] == 4:
            rgb = np.ascontiguousarray(img[..., :3])
        elif img.ndim == 3 and img.shape[0] == 4:
            rgb = np.ascontiguousarray(np.moveaxis(img[:3], 0, -1))
        else:
            skipped.append((fname, img.shape))
            print(f"[跳过] {fname}  无法识别的四通道排布 shape={img.shape}")
            del img
            continue

        tifffile.imwrite(tmp_path, rgb, photometric="rgb")
        os.replace(tmp_path, path)
        converted.append((fname, img.shape, rgb.shape))
        print(f"[转换] {fname}  {img.shape} -> {rgb.shape}")
        del img, rgb
    except Exception as e:
        failed.append((fname, str(e)))
        print(f"[失败] {fname}  {e}")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

print(f"\n完成: 转换 {len(converted)} 个，跳过 {len(skipped)} 个，失败 {len(failed)} 个")


[跳过] S003_1.tif  shape=(8432, 12000, 3)
[跳过] S004_5.tif  shape=(11240, 7171, 3)
[跳过] S007_1.tif  shape=(7497, 12096, 3)
[跳过] S008_5.tif  shape=(9541, 9953, 3)
[跳过] S011_1.tif  shape=(12691, 8969, 3)
[跳过] S012_5.tif  shape=(10744, 9972, 3)
[跳过] S016_1.tif  shape=(10342, 12827, 3)
[跳过] S016_5.tif  shape=(12690, 10997, 3)
[跳过] S019_1.tif  shape=(11576, 13496, 3)
[跳过] S021_5.tif  shape=(12835, 13186, 3)
[跳过] S023_1.tif  shape=(12744, 14161, 3)
[跳过] S024_5.tif  shape=(12556, 12799, 3)
[跳过] S027_1.tif  shape=(13221, 13893, 3)
[跳过] S028_5.tif  shape=(13131, 13208, 3)
[跳过] S029_5.tif  shape=(13967, 12840, 3)
[跳过] S030_1.tif  shape=(13270, 13915, 3)
[跳过] S031_1.tif  shape=(13392, 13719, 3)
[跳过] S034_5.tif  shape=(14520, 14691, 3)
[跳过] S035_1.tif  shape=(13392, 14083, 3)
[跳过] S036_1.tif  shape=(14214, 14791, 3)
[跳过] S039_5.tif  shape=(14467, 14294, 3)
[跳过] S041_1.tif  shape=(15099, 13731, 3)
[跳过] S045_5.tif  shape=(15177, 13474, 3)
[跳过] S047_2.tif  shape=(14144, 15028, 3)
[跳过] S048_5.tif  shape=